Import everthing needed

In [ ]:
import pandas as pd
import numpy as np
import os
import wandb
import datetime
import random
import holidays
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset, DataLoader
from imblearn.metrics import geometric_mean_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, f1_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

Set systempath

In [ ]:
import sys
sys.path.append("../src")

from utils.preprocessing import preprocess_data, prepare_data
from evaluation.comp_metrics import evaluate_all_metrics
from trainers.lstm_trainer import LSTMTrainer

Set wandb key (don't push to repository)

In [ ]:
os.environ["WANDB_API_KEY"] = "3aaf9f796df65417b3f5f8560b43875171b55805"

Set seed

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Load the data

In [ ]:
df_train = pd.read_csv("../data/classification/classification-train.csv")
df_test = pd.read_csv("C:../data/classification/classification-test.csv")

In [ ]:
print(f"Dataset shape: {df_train.shape}")
print(f"\nColumns: {df_train.columns.tolist()}")

In [ ]:
df_train = preprocess_data(df_train)

In [ ]:
df_train = prepare_data(df_train)

In [ ]:
config = {
    "hidden_size": 64,
    "num_layers": 2,
    "dropout": 0.3,
    "sequence_length": 12,
    "learning_rate": 0.0001,
    "weight_decay": 1e-4,
    "batch_size": 32,
    "num_epochs": 50,
    "gradient_clip_val": 1.0,
    "warmup_epochs": 5,
    "fp_penalty_weight": 50,
    "fp_penalty_threshold": 0.1
}

In [ ]:
trainer = LSTMTrainer(
    config=config,
    data=df_train,
    wandb_project="AICOMP_Flextrack",
    wandb_entity="fabian-dubach-hochschule-luzern",
    evaluation_fn=None
)

In [ ]:
best_val_metric = trainer.train()
print("Best validation metric:", best_val_metric)